# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This week turns the audit lens around: first on FlyRank's own research paper (*The State of AI-Driven SEO*, March 2026), then on **my** Week-5 model. Method first, verdicts never  -  the same careful reading I want applied to my own work.

## 1. Two paper findings + my methodology questions

**Method, stated before any reading:** for each finding I run the same four-step scan the lecture used:
paraphrase the claim in plain words → locate where the label/outcome comes from → deconstruct the evaluation design → mark the proof boundary (which words are backed by numbers, which drift past them). The goal is not to grade the paper  -  it discloses its own standards honestly ("No p-values or confidence intervals are reported", "Observational study: correlations do not prove causation")  -  it is to practice asking the questions I want asked of *my* model in Section 2.

---

**Finding A  -  "The Freshness Multiplier" (paper's Finding #4, tagged CONFIRMED).**

*Claim, paraphrased:* pages aged 365+ days that were refreshed within the last 30 days show a 3.2x health-score boost (10.7 → 34.5) and 57x more impressions (71 → 4,039) than the comparison baseline; refresh timing is "one of the strongest measured levers available".

*Where does the outcome come from?* Two measured quantities, both from the same local snapshot window: impressions (raw GSC data) and the **health score  -  a FlyRank composite built by a fixed recipe: 30 pts impressions + 30 pts position + 20 pts CTR + 20 pts scroll depth**. The grouping variable (refreshed within 30 days vs not) comes from a days-since-update field.

*My methodology questions:*

1. **Who gets into the refreshed group?** Refresh targets are chosen because they already have proven visibility  -  the paper's own playbook says to start with "older pages with prior visibility". So the comparison has no control group of comparable *unrefreshed* old pages. Part of the gap is plausibly selection: pages worth refreshing were already the healthier ones. What would the 57x look like against matched unrefreshed pages on the same visibility floor?
2. **Are the two headline numbers independent evidence?** Impressions are an *ingredient* of the health score (30 of 100 points), and CTR is impressions-linked on both sides of its ratio. So "3.2x health" and "57x impressions" share ingredients  -  they are partly the same measurement quoted through two formulas, not two confirmations.
3. **How fragile are the neighboring buckets?** The paper itself shows the adjacent 361+-day-fresh bucket reading 283:1 growth-to-decline on **283 growing vs 1 declining page** and correctly refuses to headline it. That is good discipline  -  and it raises the fair question of what interval the 3.2x/57x multipliers would need before carrying a "strongest lever" phrase, given none are reported.

*Constructive question I would send the authors:* could you publish the refreshed-vs-unrefreshed comparison matched on visibility floor and age, or state the bucket sizes behind the multipliers, so a reader can see how much is refresh effect and how much is selection?

---

**Finding B  -  "The Content Performance Curve" (paper's Finding #2, tagged CONFIRMED).**

*Claim, paraphrased:* health peaks at 61-90 days of age (33.1), decays to 14 by 271-365 days, then rebounds to 25.1 at 365+ days  -  read as a content lifecycle with a decay cliff.

*Where does the outcome come from?* The health score again  -  a composite of impressions, position, CTR and scroll depth  -  bucketed by content age, measured **once**, as a cross-section: every age bucket is a different set of pages observed at the same moment, not the same pages followed over time.

*My methodology questions:*

1. **Is a lifecycle curve readable off one cross-section?** The curve assumes age buckets behave like cohorts. But pages that survive to 365+ days are survivors, and the portfolio itself changed massively month to month (the paper's own trend table: active content grows from ~17K in Oct-25 to ~178K in Mar-26), so young and old buckets lived through different portfolio regimes. A cohort view  -  following one publication month forward  -  is the validation design this claim would need.
2. **Does the composite outcome carry a "performance" claim?** Health is FlyRank business logic, not an independent outcome like impressions or clicks. A decline in the recipe sum is observed; whether it equals "performance decay" in the world is an extra step the word choice glides over.
3. **Credit where due:** the paper narrows its own rebound finding ("the safe reading is narrower: older pages can recover when they are updated well. This is not evidence that age naturally reverses performance decline on its own"). That is exactly the claim-hygiene I should imitate  -  and it quietly concedes that the 365+ point is refresh-selected, connecting back to Finding A.

*Constructive question I would send the authors:* does the decay cliff appear when a single publication cohort is followed forward month by month? If yes, the lifecycle reading gets much stronger; if no, the curve is a snapshot artifact worth relabeling.

---

Both findings matter to my project personally: my Week-5 frame leans on the same staleness signal (`content_age_days`), and my label is a proxy derived from future impressions  -  so every question above has a version I must answer about my own pipeline in Sections 2-4.

In [1]:
# Paper-audit grounding, part 1: rebuild the Week-5 development frame (same tables,
# filters, features, label) so every later section runs on identical data. The frame
# builder below is Week 5's SQL, parameterized by partition so Section 2 can also build
# the shifted April frame without copying code.
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import json
from datetime import timedelta, date
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import duckdb

SEED = 42  # every stochastic step in this notebook uses this one seed

print("pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| duckdb", duckdb.__version__)

# --- token resolution: env var -> Colab secret -> repo .env -> interactive prompt ---
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN and Path("../../.env").exists():
    for line in Path("../../.env").read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            HF_TOKEN = line.split("=", 1)[1].strip()
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")


def build_frame(feature_month: str, future_month: str):
    """Week-5 frame SQL, parameterized: aggregate the feature-month partition (GSC rows),
    join the future-month partition (label window), derive the 5 contracted features +
    is_declining_next30. Boundary asserts prove partitions cover their windows."""
    fact_feat = f"read_parquet('{REL}/fact_content_daily_performance/{feature_month}/*.parquet')"
    fact_fut = f"read_parquet('{REL}/fact_content_daily_performance/{future_month}/*.parquet')"

    cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {fact_feat}").fetchone()[0]
    label_start = cutoff_date + timedelta(days=1)
    label_end = cutoff_date + timedelta(days=30)

    # Boundary check: the future partition must start exactly at the label window and
    # cover at least through its end (a 31-day month partition legitimately overshoots).
    part_min, part_max = con.sql(
        f"SELECT MIN(report_date), MAX(report_date) FROM {fact_fut}"
    ).fetchone()
    print(f"Feature month {feature_month} | cutoff {cutoff_date} | "
          f"label window {label_start} .. {label_end} | "
          f"future partition bounds {part_min} .. {part_max}")
    assert str(part_min) == str(label_start), "Future partition starts late - label window uncovered"
    assert part_max >= label_end, "Future partition ends before the label window - label window uncovered"

    frame = con.sql(f"""
        WITH recent AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS recent30_impressions,
                SUM(gsc_clicks) AS recent30_clicks,
                AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
                COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
                COUNT(DISTINCT report_date) AS recent30_days
            FROM {fact_feat}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        ),
        future AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS future30_impressions,
                COUNT(DISTINCT report_date) AS future30_days
            FROM {fact_fut}
            WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
              AND gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT
            r.client_hash_id,
            r.content_hash_id,
            r.recent30_impressions,
            LN(1 + r.recent30_impressions) AS log_recent30_impressions,
            100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
            r.recent30_avg_position,
            r.recent30_active_days,
            DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
            f.future30_impressions,
            CASE
                WHEN r.recent30_impressions >= 100
                     AND f.future30_impressions < 0.80 * r.recent30_impressions
                THEN 1 ELSE 0
            END AS is_declining_next30
        FROM recent r
        INNER JOIN future f USING (client_hash_id, content_hash_id)
        LEFT JOIN (
            SELECT client_hash_id, content_hash_id, content_created_date
            FROM {DIM_CONTENT}
        ) c USING (client_hash_id, content_hash_id)
        WHERE r.recent30_days >= 14
          AND f.future30_days >= 14
          AND r.recent30_impressions >= 100
    """).df()

    # Stable row order => deterministic ranking metrics (SQL GROUP BY order is not stable).
    frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

    meta = {
        "feature_month": feature_month,
        "future_month": future_month,
        "cutoff": str(cutoff_date),
        "label_window": [str(label_start), str(label_end)],
        "rows": int(len(frame)),
        "base_rate": float(frame["is_declining_next30"].mean()),
        "clients": int(frame["client_hash_id"].nunique()),
        "largest_client_share": float(frame["client_hash_id"].value_counts(normalize=True).iloc[0]),
    }
    return frame, meta


df_march, meta_march = build_frame("month=2026-03", "month=2026-04")
df = df_march.copy()

print(f"\nFeature frame rows: {len(df):,}")
print(f"Label distribution: {df['is_declining_next30'].mean():.1%} declining")
print(f"Cutoff date: {meta_march['cutoff']}")

assert len(df) == 96268, (
    f"Frame size {len(df):,} != Week-4/5 receipt of 96,268 - investigate before continuing"
)
print("Frame-size receipt vs Weeks 4-5: PASS")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

TARGET = "is_declining_next30"
FEATURES = [
    "log_recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "recent30_active_days",
    "content_age_days",
]

n_missing = int(df[FEATURES].isna().sum().sum())
print(f"\nMissing values across the 5 contracted features: {n_missing}")
df_model = df.dropna(subset=FEATURES).copy()
print(f"Rows available for modeling: {len(df_model):,}")


def make_baseline_scores(frame):
    """Verbatim re-implementation of the Week-4 hand rule. No fitted parameters."""
    is_visible = (frame["recent30_impressions"] >= 500).astype(int)
    is_top10 = ((frame["recent30_avg_position"] > 0) & (frame["recent30_avg_position"] <= 10)).astype(int)
    is_low_ctr = (frame["recent30_ctr_pct"] < 1.0).astype(int)
    is_stale = (frame["content_age_days"] >= 91).astype(int)
    low_ctr_top10 = is_visible * is_top10 * is_low_ctr
    visible_stale = is_visible * is_stale
    return 0.40 * is_visible + 0.35 * low_ctr_top10 + 0.25 * visible_stale


def precision_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    return float(top_k.mean()) if len(top_k) > 0 else 0.0


def recall_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    positive_total = labels.sum()
    if positive_total == 0:
        return 0.0
    return float(top_k.sum() / positive_total)


def ndcg_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    gains = (2 ** labels[order[:k]] - 1) / np.log2(np.arange(2, k + 2))
    ideal = np.sort(labels)[::-1][:k]
    ideal_gains = (2 ** ideal - 1) / np.log2(np.arange(2, k + 2))
    denom = ideal_gains.sum()
    if denom == 0:
        return 0.0
    return float(gains.sum() / denom)


models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED,
    ),
}

print("\nFrame, metric functions, hand rule and models loaded (all verbatim from Weeks 4-5).")
print("New metrics in Week 6, and why they earn a place here:")
print("- ROC-AUC: can the score separate declining from healthy pages at EVERY threshold?")
print("  Ranking metrics above only judge order near the top; ROC-AUC judges separation")
print("  across the whole range. Week 5 never reported it.")
print("- Brier score: the mean squared error of the predicted PROBABILITY against what")
print("  actually happened, 0 = perfect, lower is better; with a ~51/49 label split a")
print("  coin-flip forecast scores ~0.25. Ranking metrics cannot see miscalibration - two")
print("  models can rank identically while one calls every page '90% sure'. The moment an")
print("  editor uses a cutoff ('show me pages above 70%'), the probability values")
print("  themselves become part of the claim, so their honesty gets its own check.")
print("  Caveat kept visible: the hand rule emits scores, not probabilities, so its Brier")
print("  row is a rough diagnostic only.")

ERROR: Could not find a version that satisfies the requirement install (from versions: none)
ERROR: No matching distribution found for install

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


pandas 3.0.3 | numpy 2.5.1 | scikit-learn 1.9.0 | duckdb 1.5.4
Feature month month=2026-03 | cutoff 2026-03-31 | label window 2026-04-01 .. 2026-04-30 | future partition bounds 2026-04-01 .. 2026-04-30

Feature frame rows: 96,268
Label distribution: 51.1% declining
Cutoff date: 2026-03-31
Frame-size receipt vs Weeks 4-5: PASS

Missing values across the 5 contracted features: 0
Rows available for modeling: 96,268

Frame, metric functions, hand rule and models loaded (all verbatim from Weeks 4-5).
New metrics in Week 6, and why they earn a place here:
- ROC-AUC: can the score separate declining from healthy pages at EVERY threshold?
  Ranking metrics above only judge order near the top; ROC-AUC judges separation
  across the whole range. Week 5 never reported it.
- Brier score: the mean squared error of the predicted PROBABILITY against what
  actually happened, 0 = perfect, lower is better; with a ~51/49 label split a
  coin-flip forecast scores ~0.25. Ranking metrics cannot see miscali

In [2]:
# Paper-audit grounding, part 2.
# (a) Verify the paper's printed arithmetic reproduces from its OWN tables -
#     careful reading starts by checking the numbers that are checkable.
# (b) Hold the paper's Finding B question against MY data: decline-rate by content-age
#     bucket on the March frame, next to the paper's health-by-age curve.
print("=== (a) Arithmetic checks - the paper's claims vs its own printed tables ===")
checks = [
    # (claim, paper value, recomputed from paper's own printed inputs)
    ("Refresh health boost   34.5 / 10.7",            3.2,   34.5 / 10.7),
    ("Refresh impressions    4039 / 71",             57.0,  4039 / 71),
    ("361d-fresh bucket      283 growing : 1 declining", 283.0, 283 / 1),
]
arith_rows = []
for label, claimed, recomputed in checks:
    ok = abs(recomputed - claimed) / claimed < 0.05
    arith_rows.append({"claim": label, "paper_value": claimed,
                       "recomputed": round(recomputed, 2), "match_5pct": ok})
arith_table = pd.DataFrame(arith_rows)
print(arith_table.to_string(index=False))
assert arith_table["match_5pct"].all(), "Paper arithmetic did not reproduce - re-read before critiquing"

health_recipe = {"impressions": 30, "position": 30, "ctr": 20, "scroll_depth": 20}
direct_volume_pts = health_recipe["impressions"]
volume_linked_pts = health_recipe["impressions"] + health_recipe["ctr"]
print(f"\nHealth-score recipe (paper p.5/p.36): {health_recipe}")
print(f"-> impressions enter the composite directly ({direct_volume_pts}/100 points)")
print(f"   and CTR ({health_recipe['ctr']}/100) is impressions-linked on both sides of its ratio.")
print("-> So Finding A's '3.2x health' and '57x impressions' share ingredients: related")
print("   measurements quoted through two formulas, not two independent confirmations.")

print("\n=== (b) The Finding B question on MY data (March frame, decline probability) ===")
bins = [-1, 30, 90, 180, 365, 10 ** 9]
labels = ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
age_table = (
    df_model.assign(age_bucket=pd.cut(df_model["content_age_days"], bins=bins, labels=labels))
            .groupby("age_bucket", observed=True)
            .agg(pages=(TARGET, "size"), decline_rate=(TARGET, "mean"))
)
print(age_table.round(3).to_string())

print("\nPaper's health-by-age curve: rises to 33.1 at 61-90d, falls to 14 at 271-365d, rebounds to 25.1.")
print("My frame answers the same lifecycle question with a DIFFERENT outcome (next-30-day decline")
print("probability, not health) on a DIFFERENT population (my labeled frame, which conditions on")
print("next-month GSC coverage - disclosed in Section 3). Shape check, not a replication:")
peak_bucket = age_table["decline_rate"].idxmax()
print(f"- My decline rate peaks at {peak_bucket}, then FALLS for the oldest bucket - non-monotonic,")
print("  echoing why a single cross-section makes a shaky lifecycle story in either direction.")
print("- Observed on my data too: bucket shapes invite lifecycle readings that cohorts may not support.")

=== (a) Arithmetic checks - the paper's claims vs its own printed tables ===
                                           claim  paper_value  recomputed  match_5pct
              Refresh health boost   34.5 / 10.7          3.2        3.22        True
                Refresh impressions    4039 / 71         57.0       56.89        True
361d-fresh bucket      283 growing : 1 declining        283.0      283.00        True

Health-score recipe (paper p.5/p.36): {'impressions': 30, 'position': 30, 'ctr': 20, 'scroll_depth': 20}
-> impressions enter the composite directly (30/100 points)
   and CTR (20/100) is impressions-linked on both sides of its ratio.
-> So Finding A's '3.2x health' and '57x impressions' share ingredients: related
   measurements quoted through two formulas, not two independent confirmations.

=== (b) The Finding B question on MY data (March frame, decline probability) ===
            pages  decline_rate
age_bucket                     
0-30d        6721         0.277
31-9

## 2. My model under an honest split (before/after)

Week 5 already validated client-grouped within March. The assignment's honest-split question therefore has three rungs, and the notebook reports all three so the **gaps between them become findings**:

**Row 1  -  naive random row split (the 'before').** Exactly what the repo's reference pipeline (`scripts/03_train_model.py`) would report: a plain 80/20 `train_test_split`, clients free to appear on both sides. This answers *"can the model re-describe clients it has already seen?"*  -  the least deployment-like question, and the number I would have shipped if I had followed the starter script blindly.

**Row 2  -  client-grouped CV (Week 5's committed design).** `GroupShuffleSplit(n_splits=5, test_size=0.25, random_state=42)` on `client_hash_id`, zero client overlap asserted per fold. Answers *"does it rank pages for a client it never saw?"* Re-running it here (same seed, same sorted frame) must reproduce Week 5's committed numbers  -  a faithfulness receipt  -  now with ROC-AUC and Brier added.

**Row 3  -  time-forward out-of-time test (the 'after').** Train on the full March frame (features Mar 1-31 → label Apr 1-30), evaluate on a freshly built April frame (features Apr 1-30 → label May 1-30). Same clients on both sides  -  inherent to any time split, and precisely why it answers a *third* question the other two cannot: *"does it survive a month of drift?"*. June 2026 stays sealed, untouched by any cell in this notebook.

```
TRAIN:  features Mar 01-31 | label Apr 01-30   <- Week-5 frame (rebuilt, receipt-checked)
TEST:   features Apr 01-30 | label May 01-30   <- built below, same SQL, shifted windows
SEAL:   June                                    <- never read
```

Reporting rules: every metric sits next to its side's base rate; top-K numbers are compared as fold means with spread, never as one lucky draw (Week 4's tie-band lesson); identical metric definitions across all three rows.

In [3]:
# Row 1 - BEFORE: naive random 80/20 row split, mirroring scripts/03_train_model.py
# (plain train_test_split, no stratification, no grouping). Clients mix freely across
# the split - quantifying that mixing is the point of this row.
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss


def full_metrics(y_true, scores):
    return {
        "p20": precision_at_k(y_true, scores, 20),
        "p50": precision_at_k(y_true, scores, 50),
        "ndcg50": ndcg_at_k(y_true, scores, 50),
        "ap": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "brier": float(brier_score_loss(y_true, np.asarray(scores, dtype=float))),
    }


train_df, test_df = train_test_split(df_model, test_size=0.20, random_state=SEED)

tr_clients = set(train_df["client_hash_id"])
te_clients = set(test_df["client_hash_id"])
overlap_clients = tr_clients & te_clients

print(f"Random 80/20 row split: train={len(train_df):,} rows | test={len(test_df):,} rows")
print(f"Clients on BOTH sides: {len(overlap_clients)} of {len(tr_clients | te_clients)} "
      f"(train-only {len(tr_clients - te_clients)}, test-only {len(te_clients - tr_clients)})")
print(f"Base rates: train {train_df[TARGET].mean():.3f} | test {test_df[TARGET].mean():.3f}")

random_results = {}
base_test_scores = make_baseline_scores(test_df).to_numpy()
random_results["baseline"] = full_metrics(test_df[TARGET], base_test_scores)
for name, model in models.items():
    model.fit(train_df[FEATURES], train_df[TARGET])
    proba = model.predict_proba(test_df[FEATURES])[:, 1]
    random_results[name] = full_metrics(test_df[TARGET], proba)

print("\n=== Row 1: NAIVE RANDOM SPLIT (what the reference script would report) ===")
random_table = pd.DataFrame(random_results).T
print(random_table.round(3).to_string())
print(f"\nRead next to the grouped row below: the P@50 gap here vs there measures how much")
print(f"client memorization a naive split quietly rewards.")

Random 80/20 row split: train=77,014 rows | test=19,254 rows
Clients on BOTH sides: 36 of 40 (train-only 4, test-only 0)
Base rates: train 0.509 | test 0.516

=== Row 1: NAIVE RANDOM SPLIT (what the reference script would report) ===
                      p20   p50  ndcg50     ap  roc_auc  brier
baseline             0.65  0.48   0.537  0.522    0.499  0.420
logistic_regression  0.70  0.68   0.689  0.636    0.653  0.232
random_forest        1.00  1.00   1.000  0.732    0.734  0.209

Read next to the grouped row below: the P@50 gap here vs there measures how much
client memorization a naive split quietly rewards.


In [4]:
# Row 2 - HONEST WITHIN-MONTH: Week 5's grouped design rerun identically (same seed,
# same sorted frame) so its committed numbers reproduce - a faithfulness receipt -
# extended with ROC-AUC + Brier, which Week 5 did not report.
from sklearn.model_selection import GroupShuffleSplit

groups = df_model["client_hash_id"]
splitter = GroupShuffleSplit(n_splits=5, test_size=0.25, random_state=SEED)
fold_splits = list(splitter.split(df_model, y=df_model[TARGET], groups=groups))

comp_rows = []
for fold, (train_idx, test_idx) in enumerate(fold_splits, start=1):
    comp_rows.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "client_overlap": len(set(df_model["client_hash_id"].iloc[train_idx])
                              & set(df_model["client_hash_id"].iloc[test_idx])),
        "test_base_rate": round(float(df_model[TARGET].iloc[test_idx].mean()), 3),
    })
fold_composition = pd.DataFrame(comp_rows)
print(fold_composition.to_string(index=False))
assert fold_composition["client_overlap"].sum() == 0, "A client leaked between train and test sides"
print("Grouped-split check - no client on both sides of any fold: PASS")

cv_rows = []
for fold, (train_idx, test_idx) in enumerate(fold_splits, start=1):
    X_train, X_test = df_model[FEATURES].iloc[train_idx], df_model[FEATURES].iloc[test_idx]
    y_train, y_test = df_model[TARGET].iloc[train_idx], df_model[TARGET].iloc[test_idx]

    entry = {"fold": fold, "test_base_rate": float(y_test.mean())}
    entry["baseline"] = full_metrics(y_test, make_baseline_scores(df_model.iloc[test_idx]).to_numpy())
    for name, model in models.items():
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        entry[name] = full_metrics(y_test, proba)
    cv_rows.append(entry)
    print(f"fold {fold}: done")

cv_results = pd.DataFrame(cv_rows)

print("\n=== Row 2: CLIENT-GROUPED CV, 5 folds (Week-5 design reproduced + extended) ===")
summary_rows = []
for scorer in ["baseline", "logistic_regression", "random_forest"]:
    row = {"scorer": scorer}
    for m in ["p20", "p50", "ndcg50", "ap", "roc_auc", "brier"]:
        vals = cv_results[scorer].apply(lambda d: d[m])
        row[f"{m}_mean"] = float(vals.mean())
        row[f"{m}_sd"] = float(vals.std())
    summary_rows.append(row)
grouped_summary = pd.DataFrame(summary_rows).set_index("scorer")
print(grouped_summary.round(3).to_string())
print(f"\nMean test-fold base rate: {cv_results['test_base_rate'].mean():.3f}")

 fold  train_rows  test_rows  client_overlap  test_base_rate
    1       71678      24590               0           0.636
    2       71959      24309               0           0.421
    3       28283      67985               0           0.485
    4       70097      26171               0           0.408
    5       70797      25471               0           0.599
Grouped-split check - no client on both sides of any fold: PASS
fold 1: done
fold 2: done
fold 3: done
fold 4: done
fold 5: done

=== Row 2: CLIENT-GROUPED CV, 5 folds (Week-5 design reproduced + extended) ===
                     p20_mean  p20_sd  p50_mean  p50_sd  ndcg50_mean  ndcg50_sd  ap_mean  ap_sd  roc_auc_mean  roc_auc_sd  brier_mean  brier_sd
scorer                                                                                                                                         
baseline                 0.43   0.313     0.512   0.214        0.497      0.232    0.514  0.118         0.497       0.028       0.440   

In [5]:
# Row 3 prep - build the April-cutoff frame: features from month=2026-04, labels from
# month=2026-05. Same builder, shifted windows; June is never touched.
df_april, meta_april = build_frame("month=2026-04", "month=2026-05")
df_april_model = df_april.dropna(subset=FEATURES).copy()

print(f"\nApril frame rows: {len(df_april):,}")
print(f"Label distribution: {df_april['is_declining_next30'].mean():.1%} declining")
print(f"Cutoff date: {meta_april['cutoff']}")

# Population-shift summary: an out-of-time test compares frames one month apart, so the
# reader needs to know how much the population itself moved.
march_grains = set(zip(df_march["client_hash_id"], df_march["content_hash_id"]))
april_grains = set(zip(df_april["client_hash_id"], df_april["content_hash_id"]))
grain_overlap = len(march_grains & april_grains) / len(april_grains)

shift_table = pd.DataFrame([
    {"frame": "march (train)", **{k: meta_march[k] for k in ["rows", "base_rate", "clients", "largest_client_share"]}},
    {"frame": "april (test)", **{k: meta_april[k] for k in ["rows", "base_rate", "clients", "largest_client_share"]}},
])
print("\n=== Frame shift, March -> April ===")
print(shift_table.round(3).to_string(index=False))
print(f"Share of April-frame page-grains also present in the March frame: {grain_overlap:.1%}")

Feature month month=2026-04 | cutoff 2026-04-30 | label window 2026-05-01 .. 2026-05-30 | future partition bounds 2026-05-01 .. 2026-05-31

April frame rows: 99,279
Label distribution: 56.0% declining
Cutoff date: 2026-04-30

=== Frame shift, March -> April ===
        frame  rows  base_rate  clients  largest_client_share
march (train) 96268      0.511       40                 0.221
 april (test) 99279      0.560       44                 0.224
Share of April-frame page-grains also present in the March frame: 85.1%


In [6]:
# Row 3 - AFTER: time-forward evaluation. Fit once on ALL of March (96k rows), score
# April, compare like-for-like. Scaler refits on March only; April is transformed,
# never fitted - the Pipeline boundary is what makes this honest.
oot_results = {}
oot_base_scores = make_baseline_scores(df_april_model).to_numpy()
oot_results["baseline"] = full_metrics(df_april_model[TARGET], oot_base_scores)
for name, model in models.items():
    model.fit(df_model[FEATURES], df_model[TARGET])
    proba = model.predict_proba(df_april_model[FEATURES])[:, 1]
    oot_results[name] = full_metrics(df_april_model[TARGET], proba)

print("=== Row 3: TIME-FORWARD (train March -> test April) ===")
oot_table = pd.DataFrame(oot_results).T
print(oot_table.round(3).to_string())
print(f"\nTest-side base rate (April frame): {df_april_model[TARGET].mean():.3f}"
      f"  <-- every number above reads against THIS")

=== Row 3: TIME-FORWARD (train March -> test April) ===
                      p20   p50  ndcg50     ap  roc_auc  brier
baseline             0.50  0.44   0.453  0.559    0.511  0.410
logistic_regression  0.55  0.54   0.521  0.626    0.606  0.240
random_forest        0.60  0.54   0.604  0.588    0.582  0.252

Test-side base rate (April frame): 0.560  <-- every number above reads against THIS


In [7]:
# Consolidated before/after: three splits x three scorers x the headline metrics.
# Top-K numbers are fold means (+/- sd) for the grouped row; single-fit numbers for the
# random and time rows; base rates printed underneath so nothing floats free.
scorer_names = ["baseline", "logistic_regression", "random_forest"]

table_rows = []
for scorer in scorer_names:
    r = random_results[scorer]
    g = grouped_summary.loc[scorer]
    o = oot_results[scorer]
    table_rows.append({
        "scorer": scorer,
        "P@50_random": r["p50"],
        "P@50_grouped_mean": g["p50_mean"],
        "P@50_grouped_sd": g["p50_sd"],
        "P@50_time_fwd": o["p50"],
        "AP_random": r["ap"],
        "AP_grouped_mean": g["ap_mean"],
        "AP_time_fwd": o["ap"],
        "ROC-AUC_random": r["roc_auc"],
        "ROC-AUC_grouped_mean": g["roc_auc_mean"],
        "ROC-AUC_time_fwd": o["roc_auc"],
        "Brier_random": r["brier"],
        "Brier_grouped_mean": g["brier_mean"],
        "Brier_time_fwd": o["brier"],
    })
before_after = pd.DataFrame(table_rows).set_index("scorer")
print("=== Three-row before/after (identical metrics, identical definitions) ===")
print(before_after.round(3).to_string())
print(f"\nBase rates: random-test {test_df[TARGET].mean():.3f} | "
      f"grouped folds mean {cv_results['test_base_rate'].mean():.3f} | "
      f"time-forward test {df_april_model[TARGET].mean():.3f}")
print("Random and time rows are single fits (one draw each); the grouped row carries the spread.")

=== Three-row before/after (identical metrics, identical definitions) ===
                     P@50_random  P@50_grouped_mean  P@50_grouped_sd  P@50_time_fwd  AP_random  AP_grouped_mean  AP_time_fwd  ROC-AUC_random  ROC-AUC_grouped_mean  ROC-AUC_time_fwd  Brier_random  Brier_grouped_mean  Brier_time_fwd
scorer                                                                                                                                                                                                                                
baseline                    0.48              0.512            0.214           0.44      0.522            0.514        0.559           0.499                 0.497             0.511         0.420               0.440           0.410
logistic_regression         0.68              0.640            0.196           0.54      0.636            0.578        0.626           0.653                 0.608             0.606         0.232               0.257           0.240
ra

**Result - read the gaps between rows, not the rows themselves.**

**Gap 1, naive random -> grouped: +0.260 of pure memorization.** On the naive split, random-forest precision@50 comes out a *perfect* 1.000, against a grouped fold mean of 0.740 (sd 0.295). The mechanism is visible in the split report: 36 of the 40 clients sit on **both** sides of the naive split, so much of every test batch was already described to the model during training. The perfect head is the model re-describing familiar clients, not forecasting unfamiliar ones. Logistic regression moves the same direction, more gently (0.680 random vs 0.640 grouped). This is the number I would have shipped by copying the starter repo's split: about +0.26 of P@50 bought with client leakage instead of skill.

**Gap 2, grouped -> time-forward: -0.200, and below the coin-flip line.** Trained on all of March and scored on the freshly built April frame, RF's P@50 falls to 0.540 while the April base rate is 0.560 - the queue head no longer beats a coin flip outright. The wider metrics agree the lift thinned out: ROC-AUC slides from 0.606 (grouped mean) to 0.582; AP holds nearly flat (0.584 -> 0.588); Brier sits at 0.252, right next to the base-rate predictor's 0.246, meaning the probabilities carry little usable information out of month. LR keeps somewhat more global structure (AP 0.626, ROC-AUC 0.606) but its head precision also drops to 0.54. Read plainly: most of the March queue-head advantage did not survive one month of drift; what survives is modest, mostly-global ranking signal.

**Client-mix caveat, with this week's measured numbers.** The largest single client holds **22.1%** of frame rows. Week 5's "fold 3 is dominated by one whale client" phrasing conflated two different things: fold 3's test *draw* concentrates 67,985 rows (about 70% of the frame) across its **ten** drawn clients, but no single client owns that mass. The grouped spread reflects draw composition, not a one-client monopoly - and this audit corrects the record rather than repeating it.

Base rates printed under the consolidated table (random-test 0.516 | grouped folds 0.510 | time-forward 0.560): each P@50 reads against its own side's coin.

**Verdict (careful words, scoped):** observed across five client-grouped folds inside a single development month (March 2026, 40 clients, five contracted features), both learned models ranked the decline-review queue at or above the hand rule in every comparison (measured means: rule 0.512, LR 0.640, RF 0.740, against a 0.510 mean fold base rate). Measured out-of-time, that head-of-queue lift largely dissolved after one month (RF P@50 0.54 vs an April base of 0.56), leaving directional, mostly global ranking signal. As decision-support: the evidence supports a queue-refit-monthly tool that helps editors spend attention; it does not evidence a standing forecaster that keeps its precision across months. [FILL-1c: your own one-line takeaway for an editor, in your words]

## 3. Leakage audit

The same hunt as Week 3, now on the final feature set **and** on the parts of the pipeline Week 3 did not check: the population definition and the new April frame. Each item below maps to the skill's attack checklist; the code cells either prove it or measure it.

- **Timeline drawn.** Features aggregate the feature-month partition only; labels live strictly in the next month. Both frames' boundaries are asserted at build time  -  re-proven below for the record.
- **No label-derived or sibling features.** The contract allows exactly five features. `trend_direction` / `trend_pct` (label siblings from the starter CSV era) and `future30_impressions` (the label's own input) are checked: absent from the feature matrix even where present as helper columns. And because a leakage audit that finds nothing should distrust itself, I deliberately inject the known-leaky column and watch the score jump  -  if the harness cannot detect a planted leak, it cannot clear an honest one.
- **No product flags.** The Week-4 rule score is a baseline to beat, never an input; asserted below.
- **Population selection checked for outcome-window information.** This is the item my earlier weeks left implicit: the frame exists only for pages with ≥14 days of GSC coverage in the *outcome* month (INNER JOIN future). Pages that went quiet or lost measurement coverage are silently excluded. Not hidden anymore  -  quantified below.
- **Split grouped by the repeating entity** (Row 2) **and time-forward** (Row 3); base rates printed next to every metric; metrics computed out-of-fold throughout; June sealed; results written to a committed metrics receipt.

In [8]:
# Structural checks: the five-feature contract, forbidden names, timeline separation.
print("=== Contract: feature matrix contains EXACTLY the five contracted features ===")
expected = {
    "log_recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "recent30_active_days",
    "content_age_days",
}
assert set(FEATURES) == expected and len(FEATURES) == 5, "Feature contract drifted"
assert TARGET not in FEATURES, "Target leaked into features"
print("PASS:", sorted(FEATURES))

print("\n=== Forbidden-name scan on the modeling matrix ===")
forbidden_markers = ("trend", "future", "label", "score", "direction")
present_forbidden = [c for c in FEATURES if any(m in c.lower() for m in forbidden_markers)]
assert not present_forbidden, f"Forbidden column reached the feature matrix: {present_forbidden}"

helper_cols_present = [c for c in df_model.columns
                       if any(m in c.lower() for m in ("trend", "future"))]
print(f"No forbidden names among FEATURES: PASS")
print(f"Helper columns present in the frame but NOT in FEATURES (used for the label or")
print(f"injection tests only, never as inputs): {helper_cols_present}")

print("\n=== Timeline: feature window strictly before label window, both frames ===")
for tag, meta in [("march", meta_march), ("april", meta_april)]:
    cutoff = date.fromisoformat(meta["cutoff"])
    ls, le = (date.fromisoformat(x) for x in meta["label_window"])
    assert cutoff < ls <= le
    print(f"PASS {tag}: features ..{meta['cutoff']} | labels {ls}..{le} | "
          f"gap = {(ls - cutoff).days} day(s)")

=== Contract: feature matrix contains EXACTLY the five contracted features ===
PASS: ['content_age_days', 'log_recent30_impressions', 'recent30_active_days', 'recent30_avg_position', 'recent30_ctr_pct']

=== Forbidden-name scan on the modeling matrix ===
No forbidden names among FEATURES: PASS
Helper columns present in the frame but NOT in FEATURES (used for the label or
injection tests only, never as inputs): ['future30_impressions']

=== Timeline: feature window strictly before label window, both frames ===
PASS march: features ..2026-03-31 | labels 2026-04-01..2026-04-30 | gap = 1 day(s)
PASS april: features ..2026-04-30 | labels 2026-05-01..2026-05-30 | gap = 1 day(s)


In [9]:
# Harness sensitivity test: plant the known-leaky column (the label's own input,
# future30_impressions) and confirm the score confesses. If injecting a leak does NOT
# move the score toward 1.0, this audit's PASS verdicts would be worthless.
from sklearn.base import clone

# Keep the FULL fold frames (all helper columns); narrow to FEATURES only when fitting
# the honest model. The leaky column lives beside FEATURES in the wide frame - slicing
# to FEATURES first is what broke the injection test the first time it ran.
train_idx, test_idx = fold_splits[0]
train_wide, test_wide = df_model.iloc[train_idx], df_model.iloc[test_idx]
X_train, X_test = train_wide[FEATURES], test_wide[FEATURES]
y_train, y_test = train_wide[TARGET], test_wide[TARGET]

clean = clone(models["random_forest"]).fit(X_train, y_train)
clean_ap = float(average_precision_score(y_test, clean.predict_proba(X_test)[:, 1]))

leaky_features = FEATURES + ["future30_impressions"]
leaky = clone(models["random_forest"]).fit(train_wide[leaky_features], y_train)
leaky_ap = float(average_precision_score(y_test, leaky.predict_proba(test_wide[leaky_features])[:, 1]))

print(f"Honest 5-feature AP (grouped fold 1):        {clean_ap:.3f}")
print(f"With future30_impressions injected (6 feats): {leaky_ap:.3f}")
print(f"Week-3 deliberate-leak reference:             ~0.998")
delta = leaky_ap - clean_ap
print(f"Jump: +{delta:.3f}")
assert delta > 0.15, "Harness failed to detect a planted leak - audit tooling is broken"
print("\nHarness sensitivity: PASS - the audit can detect a leak when one exists.")
print("Column removed again; everything downstream runs on the honest five.")

Honest 5-feature AP (grouped fold 1):        0.779
With future30_impressions injected (6 feats): 0.992
Week-3 deliberate-leak reference:             ~0.998
Jump: +0.212

Harness sensitivity: PASS - the audit can detect a leak when one exists.
Column removed again; everything downstream runs on the honest five.


In [10]:
# Population-selection audit: HOW MANY candidate pages does the INNER-JOIN-future
# design exclude, and why? A page enters the frame only if the recent side passes the
# visibility/coverage floors AND the outcome month shows >= 14 days of GSC coverage.
survivorship_sql_tpl = """
    WITH recent_pool AS (
        SELECT client_hash_id, content_hash_id
        FROM {feat}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING COUNT(DISTINCT report_date) >= 14
           AND SUM(gsc_impressions) >= 100
    ),
    outcome AS (
        SELECT client_hash_id, content_hash_id,
               COUNT(DISTINCT report_date) AS outcome_days
        FROM {fut}
        WHERE report_date BETWEEN DATE '{ls}' AND DATE '{le}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        COUNT(*) AS pool_candidates,
        COUNT(*) FILTER (WHERE o.outcome_days IS NOT NULL AND o.outcome_days >= 14) AS labeled,
        COUNT(*) FILTER (WHERE o.outcome_days IS NULL) AS dropped_absent_outcome,
        COUNT(*) FILTER (WHERE o.outcome_days IS NOT NULL AND o.outcome_days < 14) AS dropped_thin_outcome
    FROM recent_pool p
    LEFT JOIN outcome o USING (client_hash_id, content_hash_id)
"""


def survivorship(meta):
    return con.sql(survivorship_sql_tpl.format(
        feat=f"read_parquet('{REL}/fact_content_daily_performance/{meta['feature_month']}/*.parquet')",
        fut=f"read_parquet('{REL}/fact_content_daily_performance/{meta['future_month']}/*.parquet')",
        ls=meta["label_window"][0], le=meta["label_window"][1],
    )).df().iloc[0]


surv_march = survivorship(meta_march)
surv_april = survivorship(meta_april)

print("=== Population selection: who never makes it into the frame? ===")
surv_table = pd.DataFrame([
    {"frame": "march->april",
     "candidates_meeting_recent_floors": int(surv_march["pool_candidates"]),
     "labeled_in_frame": int(surv_march["labeled"]),
     "dropped_no_outcome_rows": int(surv_march["dropped_absent_outcome"]),
     "dropped_outcome_lt14d": int(surv_march["dropped_thin_outcome"])},
    {"frame": "april->may",
     "candidates_meeting_recent_floors": int(surv_april["pool_candidates"]),
     "labeled_in_frame": int(surv_april["labeled"]),
     "dropped_no_outcome_rows": int(surv_april["dropped_absent_outcome"]),
     "dropped_outcome_lt14d": int(surv_april["dropped_thin_outcome"])},
])
surv_table["dropped_pct_of_candidates"] = (
    100.0 * (surv_table["candidates_meeting_recent_floors"] - surv_table["labeled_in_frame"])
    / surv_table["candidates_meeting_recent_floors"]
)
print(surv_table.to_string(index=False))

# Internal consistency: the survivorship query must reproduce the built frames exactly.
assert int(surv_march["labeled"]) == len(df_march), "Survivorship query disagrees with built March frame"
assert int(surv_april["labeled"]) == len(df_april), "Survivorship query disagrees with built April frame"
print("\nConsistency: survivorship counts equal the built frames exactly: PASS")

# ---- receipts: commit the audit's numbers ----
full_rule_scores = make_baseline_scores(df_model)
band_mask = full_rule_scores == full_rule_scores.max()

receipt = {
    "notebook": "w06_validation_audit",
    "frames": {"march": meta_march, "april": meta_april},
    "random_split": {
        "shared_clients_both_sides": int(len(overlap_clients)),
        "test_base_rate": float(test_df[TARGET].mean()),
        "metrics": {k: {m: float(v) for m, v in d.items()} for k, d in random_results.items()},
    },
    "grouped_cv": {
        "per_fold": [
            {"fold": int(r["fold"]),
             "test_base_rate": float(r["test_base_rate"]),
             **{s: {m: float(v) for m, v in r[s].items()} for s in scorer_names}}
            for _, r in cv_results.iterrows()
        ],
        "summary": {s: {m: float(v) for m, v in grouped_summary.loc[s].items()} for s in scorer_names},
    },
    "time_forward": {
        "test_base_rate": float(df_april_model[TARGET].mean()),
        "metrics": {k: {m: float(v) for m, v in d.items()} for k, d in oot_results.items()},
    },
    "leak_injection": {"honest_ap": clean_ap, "injected_ap": leaky_ap},
    "survivorship": {
        "march_to_april": {k: int(v) for k, v in surv_march.items()},
        "april_to_may": {k: int(v) for k, v in surv_april.items()},
    },
    "rule_tie_band": {"n": int(band_mask.sum()), "decline_rate": float(df_model.loc[band_mask, TARGET].mean())},
    "seed": SEED,
    "library_versions": {"pandas": pd.__version__, "numpy": np.__version__,
                         "scikit-learn": sklearn.__version__, "duckdb": duckdb.__version__},
}
out_path = Path("../../work/outputs/validation_audit_metrics.json")
out_path.write_text(json.dumps(receipt, indent=2))
print(f"\nReceipts written: {out_path}")

=== Population selection: who never makes it into the frame? ===
       frame  candidates_meeting_recent_floors  labeled_in_frame  dropped_no_outcome_rows  dropped_outcome_lt14d  dropped_pct_of_candidates
march->april                             99000             96268                      412                   2320                   2.759596
  april->may                            102943             99279                      506                   3158                   3.559251

Consistency: survivorship counts equal the built frames exactly: PASS

Receipts written: ..\..\work\outputs\validation_audit_metrics.json


**Leakage-audit verdict - checklist item by item:**

- **Timeline drawn: PASS.** Both frames assert feature-window end strictly before label-window start at build time; the gap is exactly one day (Mar 31 cutoff -> Apr 1 label start; Apr 30 -> May 1).
- **Feature contract: PASS.** The modeling matrix contains exactly the five contracted features; the target is not among them; no `trend*`, `future*`, `label`, or score-derived name reaches `FEATURES`. The one future-window column present in the wider frame (`future30_impressions`) exists only to define the label and to power the injection test below - the contract assert proves it never enters the model.
- **Harness sensitivity: PASS.** Planting that column on grouped fold 1 jumps AP from **0.779 to 0.992** (+0.212), landing next to the Week-3 deliberate-leak reference of ~0.998. A harness that catches a planted leak is one whose clean bill of sale is worth something. The column was removed again; every number in this notebook comes from the honest five.
- **No product flags: PASS.** The Week-4 rule score is computed for evaluation only and asserted absent from the features; the legacy rule remains the baseline to beat, never an input.
- **Population selection: MEASURED, NOW DISCLOSED.** The frame-building INNER JOIN keeps only pages with at least 14 days of GSC coverage in the outcome month. Measured: March keeps 96,268 of 99,000 recent-side candidates (**2.76% dropped**: 412 with no April rows at all, 2,320 with fewer than 14 covered days); April keeps 99,279 of 102,943 (**3.56% dropped**). Small in count - but the bias has a direction: pages whose measurement died are excluded, and those skew toward pages drifting out of visibility. Every statistic in this notebook therefore describes *pages measurable next month*, not all pages. That is a conditioning choice, now a stated one, and it bounds what Sections 2 and 4 may claim.
- **Splits, base rates, out-of-fold metrics, sealed June, committed receipts: PASS.** Grouped CV with zero client overlap per fold; time-forward test with June untouched; every metric printed beside its base rate; the audit's numbers written to `work/outputs/validation_audit_metrics.json`.

What the audit changes: my Week-5 numbers stand as *within-March, client-honest, next-month-measurable* results - and nothing more. Section 4 rewrites the sentence that drifted past that boundary.

## 4. Claim rewrite

The boldest sentence in my Week-5 notebook, quoted verbatim:

> "Vs the baseline: neither model ever loses a fold at precision@50 - LR wins all five outright (5/5); RF wins four and ties one (fold 3, 0.40 vs 0.40) - the lift is consistent across client draws, not a lucky fold."

**Where it goes further than the evidence:**

1. *"Never loses" and "consistent" read as properties of the world.* The evidence is five reshuffles of one month's rows - correlated resamples sharing 100% of their data, closer to one experiment reported five ways than five independent replications.
2. *No scope tags.* The sentence omits March 2026, ~40 clients, five features, inherited untuned hyperparameters - precisely the boundaries that make it true.
3. *It had not met time yet.* Week 5 flagged temporal drift as out of scope; Row 3 above is the first evidence the claim's spirit faces a month it did not train on - and the lift thinned out there (RF P@50 0.540 vs an April base of 0.560, down from a grouped mean of 0.740).

**Rewritten in safe language:**

> "Observed across five client-grouped folds within a single development month (March 2026, ~40 clients, five contracted features): both learned models scored at or above the hand rule's precision@50 in every fold - logistic regression won 5/5 outright, random forest 4/5 with one tie. Measured fold means: rule 0.512, LR 0.640, RF 0.740, against a 0.510 mean fold base rate. This is directional evidence that learned scoring orders the refresh-review queue better than the tie-bound rule under these conditions, offered as decision-support for editors' triage. It does not establish performance across months: measured out-of-time on the April frame, head-of-queue precision fell to 0.54 against a 0.56 base rate - one month, one portfolio, directional only."

A second sentence deserves the same treatment. Week 5 said: "under fold conditions the hand rule performs *at chance* on average (0.512 vs 0.510)." Tighter version: the rule's *top-50 draw* performs at chance, not the rule's signal - its maximum-score band of 24,775 pages carries a 0.532 decline rate against a 0.511 frame base rate, so the rule concentrates mild signal and then cannot rank inside it. The distinction matters: "no signal" and "signal without resolution" lead to different next steps.

**House rules going forward:** *observed* = happened in my folds/tests, no universe implied; *measured* = attached to reproducible numbers with spread; *directional* = indicates a tendency, never guarantees one; *decision-support* = helps a person allocate attention, never acts alone. Every number in the rewrites above is printed in the receipt cell below so the prose carries no unverifiable figures.

In [11]:
# Every quantity cited by the Section-4 rewrite, printed where the reader can check it.
_rule_scores_full = make_baseline_scores(df_model)
_band_mask = _rule_scores_full == _rule_scores_full.max()
band_n = int(_band_mask.sum())
band_rate = float(df_model.loc[_band_mask, TARGET].mean())
whale_share = float(df_model["client_hash_id"].value_counts(normalize=True).iloc[0])

g = grouped_summary.loc["random_forest"]
print("Quantities backing the rewritten claim:")
print(f"  Grouped P@50, RF: mean {g['p50_mean']:.3f} (sd {g['p50_sd']:.3f}) over 5 client draws, ONE month")
print(f"  Largest single client share of frame rows: {whale_share:.1%}")
print(f"  Rule tie band: n={band_n:,}, decline rate {band_rate:.3f} (frame base {meta_march['base_rate']:.3f})")
print(f"  Random-split P@50 RF: {random_results['random_forest']['p50']:.3f} "
      f"(naive-split inflation vs grouped mean: "
      f"{random_results['random_forest']['p50'] - g['p50_mean']:+.3f})")
print(f"  Time-forward P@50 RF: {oot_results['random_forest']['p50']:.3f} "
      f"(vs grouped mean: {oot_results['random_forest']['p50'] - g['p50_mean']:+.3f})")

Quantities backing the rewritten claim:
  Grouped P@50, RF: mean 0.740 (sd 0.295) over 5 client draws, ONE month
  Largest single client share of frame rows: 22.1%
  Rule tie band: n=24,775, decline rate 0.532 (frame base 0.511)
  Random-split P@50 RF: 1.000 (naive-split inflation vs grouped mean: +0.260)
  Time-forward P@50 RF: 0.540 (vs grouped mean: -0.200)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.